# Delete Exposures and Nights

Interactively delete exposures and nights of observations after inspect_nights.

Author: Qiushi Chris Tian

Created: 2026-09-28

<!-- Updated: -->

## Setting/Config

In [ ]:
TARGET = 'TOI-431'

WHOLE_NIGHTS = [20231015, ]
BAND_NIGHTS = {20240208: 'I', }
JDS = [2460237.8099118057, ]

print(
    f'{len(WHOLE_NIGHTS)} whole night(s), '
    f'{len(BAND_NIGHTS)} band night(s), and '
    f'{len(JDS)} individual exposure(s)'
)

PREVIEW_COLS = ['band', 'night', 'time', 'jd', 'airmass', 'exptime', 'skytemp']

## Preamble

In [ ]:
# %matplotlib ipympl

In [ ]:
# from dep import plt, va
import numpy as np
from pathlib import Path
from astropy.table import Table
from newport import get_input_path

In [ ]:
input_path = get_input_path(TARGET)
print(f'Reading original table from: {input_path}\n')

t = Table.read(input_path)
print(f'Original length: {len(t)}\n')

## Building Mask

In [ ]:
del_mask = np.zeros(len(ori_table), dtype=bool)

for wn in WHOLE_NIGHTS:
    selected = t['night'] == wn
    print(f'{np.sum(selected)} exposure(s) on {wn} ...')
    del_mask |= selected

for night, bands in BAND_NIGHTS.items():
    for band in bands:
        selected = (t['band'] == band) & (t['night'] == night)
        print(f'{np.sum(selected)} exposure(s) in {band} band on {night} ...')
        del_mask |= selected

for jd in JDS:
    selected = t['jd'] == jd
    print(f'1 exposure on {t[selected]["night"][0]} ...')
    del_mask |= selected

print('Done!')

## Preview Deletion

In [ ]:
dt = t[del_mask]

print(f'Number of exposure(s): {len(dt)}')
print(f'Unique night(s): {len(np.unique(dt['night']))}\n')

dt[PREVIEW_COLS].pprint_all()

print(f'\nAfter deletion, length will become {len(t) - len(dt)}')

## Executing Deletion

In [ ]:
new_t = t[~del_mask]

print(f'New length: {len(new_t)}')

## Save Table

In [ ]:
save_path = f'data/tables/opt_comp_stars/{TARGET}/{input_path.stem}_manual_del{input_path.suffix}'
print(f'Modified table will be saved to: {save_path}\n')

In [ ]:
if not Path(save_path).exists() or input(f'{save_path} exists. Overwrite? (y/*)').casefold().startswith('y'):
    new_t.write(save_path, overwrite=True)
    print(f'Saved to {save_path}')
else:
    print(f'Saving aborted')